### Imports - config.py

In [1]:
import os
import json
import time
import re
from collections import Counter
from pathlib import Path

from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_core.documents import Document

from langchain_chroma import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

C:\Users\Mohammad Zaid\AppData\Local\Temp\ipykernel_6624\2556954842.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Mohammad Zaid\Desktop\AG-Training\Project-build\ai_knowledge_assistant_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
"""GROQ_API_KEY=
GROQ_MODEL=llama-3.3-70b-versatile
EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2
VECTOR_DB=chroma
CHUNK_SIZE=1000
CHUNK_OVERLAP=150
TOP_K=5
"""

In [2]:
load_dotenv()

True

In [3]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY Error:")
os.environ["GROQ_MODEL"] = os.getenv("GROQ_MODEL")
if not os.getenv("GROQ_MODEL"):
    raise ValueError("GROQ_MODEL Error:")
os.environ["EMBEDDING_MODEL"] = os.getenv("EMBEDDING_MODEL")
if not os.getenv("EMBEDDING_MODEL"):
    raise ValueError("EMBEDDING_MODEL Error:")
os.environ["VECTOR_DB"] = os.getenv("VECTOR_DB")
if not os.getenv("VECTOR_DB"):
    raise ValueError("VECTOR_DB Error:")
os.environ["CHUNK_SIZE"] = os.getenv("CHUNK_SIZE")
if not os.getenv("CHUNK_SIZE"):
    raise ValueError("CHUNK_SIZE Error:")
os.environ["CHUNK_OVERLAP"] = os.getenv("CHUNK_OVERLAP")
if not os.getenv("CHUNK_OVERLAP"):
    raise ValueError("CHUNK_OVERLAP Error:")
os.environ["TOP_K"] = os.getenv("TOP_K")
if not os.getenv("TOP_K"):
    raise ValueError("TOP_K Error:")
print("All environment variables loaded successfully.")

All environment variables loaded successfully.


### Load PDF using PyPDF

In [4]:
file_path="./data/raw/Amazon-2025-Annual-Report.pdf"

In [5]:
from langchain_community.document_loaders import PyPDFLoader


def load_pdf(file_path):
    try:
        return PyPDFLoader(file_path).load()
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return []
    except Exception as e:
        print(f"Error loading PDF: {e}")
        return []


pdf_path = file_path
docs = load_pdf(pdf_path)


In [6]:
if docs:
    print(f"Loaded {len(docs)} pages from PDF.\n")
    print("First page content:\n", docs[0].page_content[:500], "...")
    print("\nMetadata:", docs[3].metadata)
    print("\nMetadata keys:", docs[0].metadata.keys())

Loaded 92 pages from PDF.

First page content:
 ANNUAL REPORT
2 0 2 5 ...

Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2022-02-14T21:08:55-06:00', 'author': 'T&C Composition', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'keywords': '26-3815-1_3', 'moddate': '2026-04-07T16:34:29-05:00', 'subject': 'Annual Report', 'title': 'Amazon.com, Inc', 'trapped': '/False', 'source': './data/raw/Amazon-2025-Annual-Report.pdf', 'total_pages': 92, 'page': 3, 'page_label': '4'}

Metadata keys: dict_keys(['producer', 'creator', 'creationdate', 'author', 'gts_pdfxconformance', 'gts_pdfxversion', 'keywords', 'moddate', 'subject', 'title', 'trapped', 'source', 'total_pages', 'page', 'page_label'])


### text cleaning before splitting
- Minimum preprocessing should include:
1. Remove empty lines.
2. Normalize excessive whitespace.
3. Preserve section headings where possible.
4. Remove repeated headers or footers if they create noise.
5. Keep metadata such as source file and page number.

- Chunking choice: `chunk_size=1000` keeps a compact section together for retrieval, while `chunk_overlap=150` preserves boundary context so headings and trailing sentences do not get separated from the next chunk.
- This is a conservative setting for annual reports and similar PDFs, where pages often mix headings, short paragraphs, and table-like text.

In [7]:
def clean_document(text):
    lines = [" ".join(line.split()) for line in text.splitlines() if line.strip()]
    return "\n".join(lines)


def set_document_metadata(document, file_path, chunk_id=""):
    source_file = Path(file_path).name
    year_match = re.search(r"(19|20)\d{2}", source_file)
    year = year_match.group(0) if year_match else ""
    metadata = dict(document.metadata or {})
    page_number = metadata.get("page_number")
    if page_number is None:
        page = metadata.get("page")
        page_number = page + 1 if isinstance(page, int) else metadata.get("page_label") or 1
    if isinstance(page_number, str) and page_number.isdigit():
        page_number = int(page_number)
    metadata.update(
        {
            "source_file": source_file,
            "page_number": page_number,
            "document_type": "pdf",
            "year": year,
            "chunk_id": chunk_id,
        }
    )
    return Document(page_content=clean_document(document.page_content), metadata=metadata)

In [8]:
cleaned_docs = [
    set_document_metadata(document, pdf_path)
    for document in docs
]

if cleaned_docs:
    print("First cleaned page content:\n", cleaned_docs[0].page_content[:500], "...")
    print("\nMetadata:", cleaned_docs[0].metadata)
else:
    print("No documents were loaded.")

First cleaned page content:
 ANNUAL REPORT
2 0 2 5 ...

Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2022-02-14T21:08:55-06:00', 'author': 'T&C Composition', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'keywords': '26-3815-1_3', 'moddate': '2026-04-07T16:34:29-05:00', 'subject': 'Annual Report', 'title': 'Amazon.com, Inc', 'trapped': '/False', 'source': './data/raw/Amazon-2025-Annual-Report.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1', 'source_file': 'Amazon-2025-Annual-Report.pdf', 'page_number': 1, 'document_type': 'pdf', 'year': '2025', 'chunk_id': ''}


In [9]:
chunk_size = int(os.getenv("CHUNK_SIZE", "1000"))
chunk_overlap = int(os.getenv("CHUNK_OVERLAP", "150"))
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

chunk_documents = text_splitter.split_documents(cleaned_docs)
chunks = []

for i, chunk_document in enumerate(chunk_documents, start=1):
    metadata = dict(chunk_document.metadata or {})
    metadata["chunk_id"] = f"chunk_{i}"
    metadata["text"] = chunk_document.page_content
    chunks.append(Document(page_content=chunk_document.page_content, metadata=metadata))

print(f"Loaded {len(cleaned_docs)} cleaned pages.")
print(f"Total chunks after splitting: {len(chunks)}")


Loaded 92 cleaned pages.
Total chunks after splitting: 421


In [10]:
if chunks:
    print(chunks[0].metadata)
    print(chunks[1].metadata)
    print(chunks[2].page_content[:500], "...")
for i in range(min(10, len(chunks))):
    print(chunks[i].metadata["chunk_id"])

{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2022-02-14T21:08:55-06:00', 'author': 'T&C Composition', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'keywords': '26-3815-1_3', 'moddate': '2026-04-07T16:34:29-05:00', 'subject': 'Annual Report', 'title': 'Amazon.com, Inc', 'trapped': '/False', 'source': './data/raw/Amazon-2025-Annual-Report.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1', 'source_file': 'Amazon-2025-Annual-Report.pdf', 'page_number': 1, 'document_type': 'pdf', 'year': '2025', 'chunk_id': 'chunk_1', 'text': 'ANNUAL REPORT\n2 0 2 5'}
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2022-02-14T21:08:55-06:00', 'author': 'T&C Composition', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'keywords': '26-3815-1_3', 'moddate': '2026-04-07T16:34:29-05:00', 'subject': 'Annual Report', 'title': 'Amazon.com, Inc',

In [11]:
file_path="./data/raw/Amazon-2025-Annual-Report.pdf"
persist_directory="./data/vector_store"
chunk_collection_name="amazon-chunks"


### Client

In [33]:
llm = ChatGroq(model=os.getenv("GROQ_MODEL"))

In [18]:
import chromadb

In [19]:

chromadb_client = chromadb.PersistentClient(
    path=persist_directory
)

In [20]:
chromadb_client

In [21]:
embedding_model = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL"))

C:\Users\Mohammad Zaid\AppData\Local\Temp\ipykernel_6624\2079451105.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL"))
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7230.47it/s]


In [22]:

vector_db = Chroma(collection_name=chunk_collection_name,
                   collection_metadata={"hnsw:space": "cosine"}, 
                   embedding_function=embedding_model,
                   client=chromadb_client, 
                   persist_directory=persist_directory
                   )

In [23]:
# def add_chunks_to_vector_db(chunks, vector_db):
if chunks:
    existing_count = vector_db._collection.count()
    if existing_count == 0:
        vector_db.add_documents(
            documents=chunks,
            ids=[chunk.metadata["chunk_id"] for chunk in chunks],
        )
        print(f"Added {len(chunks)} chunks to Chroma collection '{chunk_collection_name}'.")
    else:
        print(f"Chroma collection '{chunk_collection_name}' already contains {existing_count} chunks.")
else:
    print("No chunks available to index.")


Chroma collection 'amazon-chunks' already contains 421 chunks.


### Retriever

In [24]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": int(os.getenv("TOP_K", "5"))}
)

In [25]:
# Test retrieval
query = "What are the key financial highlights of Amazon in 2025?"
retrieved_chunks = retriever.invoke(query)
print(f"Retrieved {len(retrieved_chunks)} chunks for the query: '{query}'\n")
for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"Chunk {i} Metadata: {chunk.metadata}")
    print(f"Chunk {i} Content:\n{chunk.page_content[:500]}...\n")

Retrieved 5 chunks for the query: 'What are the key financial highlights of Amazon in 2025?'

Chunk 1 Metadata: {'source': './data/raw/Amazon-2025-Annual-Report.pdf', 'keywords': '26-3815-1_3', 'chunk_id': 'chunk_221', 'creationdate': '2022-02-14T21:08:55-06:00', 'trapped': '/False', 'year': '2025', 'author': 'T&C Composition', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'moddate': '2026-04-07T16:34:29-05:00', 'producer': 'Adobe PDF Library 15.0', 'subject': 'Annual Report', 'text': 'Guidance\nWe provided guidance on February 5, 2026, in our earnings release furnished on Form 8-K as set forth below. These\nforward-looking statements reflect Amazon.com’s expectations as of February 5, 2026, and are subject to substantial\nuncertainty. Our results are inherently unpredictable and may be materially affected by many factors, such as fluctuations in\nforeign exchange rates and energy prices, changes in global economic and geopolitical conditions, tariff and trade policies,\nresource and s

In [28]:
# build context from retrieved chunks
context_citations = [f"Source: {chunk.metadata['source_file']} (Page {chunk.metadata['page_number']})" for chunk in retrieved_chunks]
context = "\n\n".join([chunk.page_content for chunk in retrieved_chunks])

In [30]:
print(context_citations)
print(context)


['Source: Amazon-2025-Annual-Report.pdf (Page 41)', 'Source: Amazon-2025-Annual-Report.pdf (Page 41)', 'Source: Amazon-2025-Annual-Report.pdf (Page 8)', 'Source: Amazon-2025-Annual-Report.pdf (Page 11)', 'Source: Amazon-2025-Annual-Report.pdf (Page 48)']
Guidance
We provided guidance on February 5, 2026, in our earnings release furnished on Form 8-K as set forth below. These
forward-looking statements reflect Amazon.com’s expectations as of February 5, 2026, and are subject to substantial
uncertainty. Our results are inherently unpredictable and may be materially affected by many factors, such as fluctuations in
foreign exchange rates and energy prices, changes in global economic and geopolitical conditions, tariff and trade policies,
resource and supply volatility, including for memory chips, and customer demand and spending (including the impact of
recessionary fears), inflation, interest rates, regional labor market constraints, world events, the rate of growth of the internet,
onli

In [ ]:
system_prompt = """

You are an enterprise knowledge assistant.
Answer the user question using only the provided context.
Rules:
- Do not use outside knowledge.
- If the answer is not available in the context, say: "I could not find this in the provided documents."
- Cite the source file and page number or chunk ID for each key claim.
- The citations are provided.
- Do not invent numbers, dates, risks, or business conclusions.
- Keep the answer clear and business-friendly.

Question:
{question}

Retrieved Context:
{context}

Citations:
{citations}

Return:
1. Answer
2. Supporting Evidence
3. Sources
4. Confidence: High / Medium / Low

"""

In [ ]:
"""The application must save query history.
Recommended format:
{
 "timestamp": "",
 "question": "",
 "query_type": "",
 "retrieved_sources": [],
 "answer": "",
 "answerability": "",
 "confidence": ""
}

Store logs in:
logs/query_logs.jsonl"""

In [32]:
from groq import Groq
groq_client = Groq()

In [37]:
prompt = [
    {
        "role": "system",
        "content": system_prompt.format(
            question=query,
            context=context,
            citations=context_citations
        )
    },
    {
        "role": "user",
        "content": query
    }
]



In [38]:
messages = [
    ("system", system_prompt.format(
        question=query,
        context=context,
        citations=context_citations
    )),
    ("user", query)
]

In [39]:
response = llm.invoke(messages)

In [41]:
response.content

"1. Direct Answer: \nThe key financial highlights of Amazon in 2025 include a 12% year-over-year revenue growth from $638 billion to $717 billion, a 17% year-over-year increase in operating income from $69 billion to $80 billion, and a decrease in free cash flow from $38 billion to $11 billion.\n\n2. Supporting Evidence:\n- Revenue grew 12% year-over-year from $638 billion to $717 billion.\n- Operating income improved 17% year-over-year from $69 billion to $80 billion, with an operating margin of 11.2%.\n- Free Cash Flow decreased from $38 billion to $11 billion due to increased purchases of property and equipment.\n\n3. Sources:\n- 'Source: Amazon-2025-Annual-Report.pdf (Page 41)'\n- 'Source: Amazon-2025-Annual-Report.pdf (Page 8)'\n\n4. Confidence: High"

In [44]:
# final answer:
response = groq_client.chat.completions.create(
    model=os.getenv("GROQ_MODEL"),
    messages=prompt,
    temperature=0.2
)
response.choices[0].message.content

"1. Direct Answer: \nThe key financial highlights of Amazon in 2025 include a 12% year-over-year revenue growth from $638 billion to $717 billion, a 17% year-over-year increase in operating income from $69 billion to $80 billion, and a decrease in free cash flow from $38 billion to $11 billion.\n\n2. Supporting Evidence:\n- Revenue grew 12% year-over-year from $638 billion to $717 billion.\n- Operating income improved 17% year-over-year from $69 billion to $80 billion, with an operating margin of 11.2%.\n- Free Cash Flow decreased from $38 billion to $11 billion, primarily due to a $50.7 billion increase in purchases of property and equipment.\n\n3. Sources:\n- 'Source: Amazon-2025-Annual-Report.pdf (Page 41)'\n- 'Source: Amazon-2025-Annual-Report.pdf (Page 8)'\n\n4. Confidence: High"